In [3]:
pip install fastapi uvicorn pandas numpy scikit-learn joblib

  Using cached fastapi-0.142.2-py3-none-any.whl.metadata (27 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached starlette-1.7.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached opentelemetry_api-1.45.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached pydantic_core-2.46.5-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
Using cached fastapi-0.142.2-py3-none-any.whl (144 kB)
Using cached uvicorn-0.54.0-py3-none-any.whl (87 kB)
Using cached annotated_doc-0.0.5-py3-none-any.whl (5.3 kB)
Using cached opentelemetry_api-1.45.0-py3-none-any.whl (60 kB)
Using cached pydantic-2.13.5-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.5-cp312-cp312-win_amd64.whl (2.0 MB)
Using cach

In [12]:
pip install --upgrade typing_extensions

Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install --upgrade fastapi pydantic pydantic-core uvicorn

Note: you may need to restart the kernel to use updated packages.


In [18]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import joblib
import os


# --------------------------------------------------
# Create FastAPI application
# --------------------------------------------------

app = FastAPI(
    title="SmartGridSense ML API",
    description="AI service for energy prediction and anomaly detection",
    version="1.0"
)


# --------------------------------------------------
# Project paths
# --------------------------------------------------

BASE_DIR = os.path.dirname(
    os.path.dirname(
        os.path.abspath(__file__)
    )
)

MODEL_DIR = os.path.join(
    BASE_DIR,
    "models"
)


# --------------------------------------------------
# Model paths
# --------------------------------------------------

prediction_model_path = os.path.join(
    MODEL_DIR,
    "energy_prediction_model.pkl"
)

prediction_features_path = os.path.join(
    MODEL_DIR,
    "features.pkl"
)

anomaly_model_path = os.path.join(
    MODEL_DIR,
    "anomaly_model.pkl"
)

anomaly_features_path = os.path.join(
    MODEL_DIR,
    "anomaly_features.pkl"
)


# --------------------------------------------------
# Load prediction model
# --------------------------------------------------

prediction_model = joblib.load(
    prediction_model_path
)

prediction_features = joblib.load(
    prediction_features_path
)


# --------------------------------------------------
# Load anomaly model
# --------------------------------------------------

anomaly_model = joblib.load(
    anomaly_model_path
)

anomaly_features = joblib.load(
    anomaly_features_path
)


# --------------------------------------------------
# Request model
# --------------------------------------------------

class EnergyRequest(BaseModel):

    data: dict


# --------------------------------------------------
# Home endpoint
# --------------------------------------------------

@app.get("/")
def home():

    return {
        "message": "SmartGridSense ML API is running",
        "prediction_model": "loaded",
        "anomaly_model": "loaded"
    }


# --------------------------------------------------
# Energy prediction
# --------------------------------------------------

@app.post("/predict")
def predict_energy(request: EnergyRequest):

    try:

        input_data = request.data

        missing_features = [
            feature
            for feature in prediction_features
            if feature not in input_data
        ]

        if missing_features:

            raise HTTPException(
                status_code=400,
                detail={
                    "message": "Missing prediction features",
                    "missing_features": missing_features
                }
            )

        row = {
            feature: input_data[feature]
            for feature in prediction_features
        }

        dataframe = pd.DataFrame([row])

        prediction = prediction_model.predict(
            dataframe
        )

        return {
            "success": True,
            "predicted_energy": float(prediction[0])
        }

    except HTTPException:
        raise

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=str(e)
        )


# --------------------------------------------------
# Anomaly detection
# --------------------------------------------------

@app.post("/detect-anomaly")
def detect_anomaly(request: EnergyRequest):

    try:

        input_data = request.data

        missing_features = [
            feature
            for feature in anomaly_features
            if feature not in input_data
        ]

        if missing_features:

            raise HTTPException(
                status_code=400,
                detail={
                    "message": "Missing anomaly features",
                    "missing_features": missing_features
                }
            )

        row = {
            feature: input_data[feature]
            for feature in anomaly_features
        }

        dataframe = pd.DataFrame([row])

        prediction = anomaly_model.predict(
            dataframe
        )

        if prediction[0] == -1:

            result = "Anomaly"

        else:

            result = "Normal"

        return {
            "success": True,
            "result": result
        }

    except HTTPException:
        raise

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

ImportError: cannot import name 'Sentinel' from 'typing_extensions' (C:\Users\sonali karnwar\anaconda3\anaconda\Lib\site-packages\typing_extensions.py)